# Sign → speech, last stage: text-to-speech options

**What this notebook does.** Diagnostic, TODO §12.6 ("TTS: pick a model, measure
latency"). The English sentence from gloss → English has to be spoken. Candidates:

| engine | where it runs | cost |
|---|---|---|
| `sapi:<voice>` | **client**: these are the Windows OS voices a browser's Web Speech API (`speechSynthesis`) uses on Windows | free, offline |
| `@cf/myshell-ai/melotts` | edge (Workers AI) | $0.0002 / audio minute |
| `@cf/deepgram/aura-1` | edge | $0.015 / 1k characters |
| `@cf/deepgram/aura-2-en` | edge | $0.030 / 1k characters |

The account is on the **Workers Free plan** (10,000 neurons/day, $0.011 per 1k neurons
beyond). At these prices one ~45-character sentence costs about **120 neurons with
Aura-2**, about 60 with Aura-1 and under 1 with MeloTTS. On the free allowance that is
roughly **80 Aura-2 sentences a day**, or thousands with MeloTTS. §2 estimates the cost
before calling and refuses to exceed `neuron_budget`.

**Measures, per engine:**
- **latency**: wall time to the complete audio (REST, not streamed). For SAPI this
  includes PowerShell start-up (~0.2 s), which a browser does not pay;
- **real-time factor** = latency / audio length;
- **intelligibility**: Whisper (`asr_model`) transcribes the audio back, and WER is taken
  against the input text (numbers spelled out on both sides). It is a proxy, since
  Whisper is more forgiving than a listener, but it catches garbled or truncated audio;
- **cost** per sentence and **free-plan sentences per day**.

**Texts:** the first English reference of each sentence in `sb.rescore`
`evalset/gloss2en.v1.jsonl`, the kind of sentence stage 3 produces. SAPI speaks all 132.
Workers AI speaks a seeded `workers_sample` (20) to stay inside the budget.

**Artifacts** (`data/cache/gislr/downstream/tts/`): `audio/<engine>/…`, `synth_<engine>.json`
(latency per text), `asr_<engine>.json` (transcripts), `results.json`, `results_table.csv`.
Resumable: existing audio and transcripts are skipped.

## Setup

In [1]:
# ============================================================
# Imports, config, texts, paths
# ============================================================
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Audio, display
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR
from sb.rescore import client as C
from sb.synthesize import asr, metrics, tts

CONFIG = json.loads((EXPERIMENTS_DIR / "recognition" / "configs" / "gislr.tts.json").read_text(encoding="utf-8"))
OUT = CACHE_DIR / "gislr" / "downstream" / "tts"
AUDIO = OUT / "audio"
AUDIO.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, obj) -> None:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=float), encoding="utf-8")
    os.replace(tmp, path)


def slug(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")


EVAL = [json.loads(l) for l in (Path(C.__file__).parent / "evalset" / f"{CONFIG['evalset']}.jsonl")
        .read_text(encoding="utf-8").splitlines() if l.strip()]
TEXTS = {r["id"]: r["refs"][0] for r in EVAL}
rng = np.random.default_rng(CONFIG["seed"])
WORKERS_IDS = sorted(rng.choice(sorted(TEXTS), CONFIG["workers_sample"], replace=False).tolist())
VOICES = tts.sapi_voices() if CONFIG["sapi_voices"] == "all" else CONFIG["sapi_voices"]
print(f"{len(TEXTS)} texts · SAPI voices {VOICES} · Workers AI sample {len(WORKERS_IDS)} · "
      f"credentials {'yes' if C.credentials() else 'no'}")

132 texts · SAPI voices ['Microsoft David Desktop', 'Microsoft Zira Desktop'] · Workers AI sample 20 · credentials no


## 1. Client-side voices (SAPI ≈ the browser's Web Speech API on Windows)

In [2]:
# ============================================================
# SAPI: every voice x every text
# ============================================================
for v in VOICES:
    eng = f"sapi-{slug(v)}"
    path = OUT / f"synth_{eng}.json"
    done = json.loads(path.read_text()) if path.exists() else {}
    for tid, text in tqdm([(k, t) for k, t in TEXTS.items() if k not in done], desc=eng):
        info = tts.sapi_speak(text, AUDIO / eng / f"{tid}.wav", v)
        done[tid] = {"path": info["path"], "latency_s": info["seconds"], "audio_s": info["audio_seconds"],
                     "chars": len(text)}
        write_json(path, done)
print({p.stem: len(json.loads(p.read_text())) for p in OUT.glob("synth_*.json")})

sapi-microsoft-david-desktop:   0%|          | 0/132 [00:00<?, ?it/s]

sapi-microsoft-zira-desktop:   0%|          | 0/132 [00:00<?, ?it/s]

{'synth_sapi-microsoft-david-desktop': 132, 'synth_sapi-microsoft-zira-desktop': 132}


## 2. Workers AI TTS (skipped without credentials)

The cost estimate uses the config's prices. MeloTTS is priced per audio minute, so it
assumes 1 s of audio per 15 characters. The call is refused if the estimate exceeds
`neuron_budget` (keep a margin for the gloss → English notebook's LLM calls on the same
day).

In [3]:
# ============================================================
# Workers AI: estimate neurons, then speak the sample, cached
# ============================================================
def est_usd(model: str, text: str) -> float:
    pr = CONFIG["pricing_usd"][model]
    if "per_1k_chars" in pr:
        return len(text) / 1000 * pr["per_1k_chars"]
    return (len(text) / 15) / 60 * pr["per_audio_min"]


def to_neurons(usd: float) -> float:
    return usd / CONFIG["usd_per_1k_neurons"] * 1000


plan = [(m, tid) for m in CONFIG["workers_models"] for tid in WORKERS_IDS
        if tid not in (json.loads((OUT / f"synth_{slug(m['model'])}.json").read_text())
                       if (OUT / f"synth_{slug(m['model'])}.json").exists() else {})]
need = sum(to_neurons(est_usd(m["model"], TEXTS[t])) for m, t in plan)
print(f"{len(plan)} uncached calls, estimated {need:.0f} neurons (budget {CONFIG['neuron_budget']})")

if C.credentials() is None:
    print("No Workers AI credentials: set CLOUDFLARE_ACCOUNT_ID and CLOUDFLARE_API_TOKEN in the repo .env.")
elif need > CONFIG["neuron_budget"]:
    print("estimate exceeds neuron_budget -- lower workers_sample or drop a model")
else:
    for m in CONFIG["workers_models"]:
        eng = slug(m["model"])
        path = OUT / f"synth_{eng}.json"
        done = json.loads(path.read_text()) if path.exists() else {}
        client = C.WorkersAITTS(m["model"], AUDIO / eng, **m["params"])
        for tid in tqdm([t for t in WORKERS_IDS if t not in done], desc=m["model"]):
            r = client.speak(TEXTS[tid])
            a = asr.load_audio(r["path"])
            done[tid] = {"path": r["path"], "latency_s": r["latency_s"], "audio_s": len(a) / asr.SAMPLE_RATE,
                         "chars": len(TEXTS[tid]), "bytes": r["bytes"]}
            write_json(path, done)

60 uncached calls, estimated 1640 neurons (budget 5000)
No Workers AI credentials: set CLOUDFLARE_ACCOUNT_ID and CLOUDFLARE_API_TOKEN in the repo .env.


## 3. Intelligibility: Whisper transcribes every clip back

In [4]:
# ============================================================
# ASR round trip per engine (cached per clip)
# ============================================================
tr = None
for sp in sorted(OUT.glob("synth_*.json")):
    eng = sp.stem.removeprefix("synth_")
    synth = json.loads(sp.read_text())
    path = OUT / f"asr_{eng}.json"
    done = json.loads(path.read_text()) if path.exists() else {}
    todo = [t for t in synth if t not in done]
    if todo:
        tr = tr or asr.Transcriber(CONFIG["asr_model"], transcribe_kwargs={"language": "en", "beam_size": 5,
                                                                          "temperature": 0.0, "vad_filter": True})
        for tid in tqdm(todo, desc=f"ASR {eng}"):
            done[tid] = tr(synth[tid]["path"]).text
            write_json(path, done)
del tr

ASR sapi-microsoft-david-desktop:   0%|          | 0/132 [00:00<?, ?it/s]

ASR sapi-microsoft-zira-desktop:   0%|          | 0/132 [00:00<?, ?it/s]

## 4. Results

In [5]:
# ============================================================
# Latency, real-time factor, intelligibility, cost per engine
# ============================================================
def norm(t: str) -> str:
    return metrics.normalize_text(asr.spell_numbers(t))


rows = []
for sp in sorted(OUT.glob("synth_*.json")):
    eng = sp.stem.removeprefix("synth_")
    synth = json.loads(sp.read_text())
    heard = json.loads((OUT / f"asr_{eng}.json").read_text()) if (OUT / f"asr_{eng}.json").exists() else {}
    ids = [t for t in synth if t in heard]
    lat = np.array([synth[t]["latency_s"] or np.nan for t in synth], float)
    aud = np.array([synth[t]["audio_s"] for t in synth], float)
    model = next((m["model"] for m in CONFIG["workers_models"] if slug(m["model"]) == eng), None)
    usd = float(np.mean([est_usd(model, TEXTS[t]) for t in synth])) if model else 0.0
    rows.append({"engine": eng, "where": "edge (Workers AI)" if model else "client (OS voice)", "n": len(synth),
                 "latency_median_s": float(np.nanmedian(lat)), "latency_p90_s": float(np.nanpercentile(lat, 90)),
                 "rtf_median": float(np.nanmedian(lat / aud)), "audio_s_median": float(np.median(aud)),
                 "asr_wer": metrics.asr_wer([norm(TEXTS[t]) for t in ids], [norm(heard[t]) for t in ids]) if ids else np.nan,
                 "usd_per_sentence": usd, "neurons_per_sentence": to_neurons(usd),
                 "free_plan_sentences_per_day": (CONFIG["free_neurons_per_day"] / to_neurons(usd)) if usd else np.inf})
TABLE = pd.DataFrame(rows).sort_values("asr_wer")
write_json(OUT / "results.json", {"results": rows, "workers_sample": WORKERS_IDS})
TABLE.to_csv(OUT / "results_table.csv", index=False)
TABLE.round(3)

,engine,where,n,latency_median_s,latency_p90_s,rtf_median,audio_s_median,asr_wer,usd_per_sentence,neurons_per_sentence,free_plan_sentences_per_day
1,sapi-microsoft-zira-desktop,client (OS voice),132,0.202,0.206,0.094,2.172,0.000,0.0,0.0,inf
0,sapi-microsoft-david-desktop,client (OS voice),132,0.198,0.201,0.091,2.182,0.003,0.0,0.0,inf


## 5. Listen

In [6]:
# ============================================================
# The same few sentences from every engine
# ============================================================
N_LISTEN = 3

ids = WORKERS_IDS[:N_LISTEN]
for tid in ids:
    print(f"\n{tid}: {TEXTS[tid]}")
    for sp in sorted(OUT.glob("synth_*.json")):
        synth = json.loads(sp.read_text())
        if tid in synth:
            print(f"  {sp.stem.removeprefix('synth_')}")
            display(Audio(synth[tid]["path"]))


animals-151: Whose dog is that?
  sapi-microsoft-david-desktop


  sapi-microsoft-zira-desktop



animals-153: The dog finished its food.
  sapi-microsoft-david-desktop


  sapi-microsoft-zira-desktop



body-054: The girl's hair is yellow.
  sapi-microsoft-david-desktop


  sapi-microsoft-zira-desktop


## 6. Reading the results

- If a client-side voice is intelligible (WER near the hosted models'), the **free,
  zero-latency default is the browser's `speechSynthesis`**. Workers AI TTS becomes an
  opt-in "natural voice", budgeted per day on the Free plan.
- Aura-2's per-character price makes it the first thing to run out on the Free plan
  (about 80 sentences a day). MeloTTS is effectively free.
- Latency here is time to the *complete* clip. A streamed response (Aura supports it
  through the binding) starts playing sooner. Measure that from `apps/edge`, not here.